<div style="background: linear-gradient(135deg, #1e3a8a 0%, #3b82f6 100%); padding: 30px; border-radius: 15px; text-align: center; color: white; box-shadow: 0 10px 20px rgba(0,0,0,0.2); margin-bottom: 20px;">
    <h1 style="margin: 0; font-size: 2.5em; font-family: 'Segoe UI', Tahoma, Geneva, Verdana, sans-serif; font-weight: 800; letter-spacing: 2px; text-transform: uppercase;">
        Disaster Tweets Analysis
    </h1>
    <hr style="border: 0; height: 1px; background-image: linear-gradient(to right, rgba(255, 255, 255, 0), rgba(255, 255, 255, 0.75), rgba(255, 255, 255, 0)); margin: 15px 0;">
    <h2 style="margin: 0; font-size: 1.5em; opacity: 0.9; font-weight: 400; font-style: italic;">
        Fine-tuning du modèle BERT
    </h2>
</div>

<h3 style="color: #1e40af; border-bottom: 2px solid #3b82f6; padding-bottom: 8px; margin-top: 25px; font-weight: bold; font-family: 'Segoe UI', sans-serif;">
    Importation des bibliothèques
</h3>


In [2]:
import builtins
import pandas as pd
import re
# Suivi des expriences (MLflow & DagsHub)
import dagshub
import mlflow

# Scikit-Learn : Sparation des donnes et mtriques d'valuation
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    accuracy_score, 
    f1_score, 
    fbeta_score, 
    precision_score, 
    recall_score
)

# TensorFlow / Keras : Cration et entranement du modle Deep Learning
import tensorflow as tf

<h3 style="color: #1e40af; border-bottom: 2px solid #3b82f6; padding-bottom: 8px; margin-top: 25px; font-weight: bold; font-family: 'Segoe UI', sans-serif;">
    DagsHub & MLflow Init
</h3>


In [3]:

# Initialisation de la connexion DagsHub avec les identifiants de votre dépôt et activation du mode MLflow
# PATCH WINDOWS : Force l'utilisation de l'encodage UTF-8 lors de l'écriture des fichiers.
# Ceci corrige l'erreur "charmap codec can't encode characters" causée par le nouveau format d'affichage (summary) de Keras 3
_original_open = builtins.open
def _utf8_open(*args, **kwargs):
    mode = kwargs.get('mode', args[1] if len(args) > 1 else 'r')
    if 'b' not in mode and 'encoding' not in kwargs:
        kwargs['encoding'] = 'utf-8'
    return _original_open(*args, **kwargs)
builtins.open = _utf8_open

dagshub.init(repo_owner='Oscar-AS', repo_name='disaster-tweets-project', mlflow=True)

# Définition du nom du dossier (expérience) dans MLflow où toutes nos métriques seront classées
mlflow.set_experiment("Disaster_Tweets")

# Affichage d'un message console pour confirmer que le tracking est bien connecté
print("MLflow activé avec succès sur DagsHub !")


Accessing as Oscar-AS

Initialized MLflow to track repo "Oscar-AS/disaster-tweets-project"

Repository Oscar-AS/disaster-tweets-project initialized!

MLflow activé avec succès sur DagsHub !


<h3 style="color: #1e40af; border-bottom: 2px solid #3b82f6; padding-bottom: 8px; margin-top: 25px; font-weight: bold; font-family: 'Segoe UI', sans-serif;">
    Importation des données
</h3>


In [4]:
# Chargement des données
# Lecture du fichier CSV depuis le dossier Base et stockage dans la variable 'df'
df = pd.read_csv("../Base/tweets.csv")



<h3 style="color: #1e40af; border-bottom: 2px solid #3b82f6; padding-bottom: 8px; margin-top: 25px; font-weight: bold; font-family: 'Segoe UI', sans-serif;">
    Nettoyage
</h3>


In [5]:
def cleaning_simple(data):
    text = str(data)  # convertir en texte pour éviter les erreurs avec les valeurs manquantes
    
    text = re.sub(r'https?://\S+|www\.\S+|http?://\S+', ' ', text)  # supprimer les URLs
    
    text = re.sub(r'<.*?>', ' ', text)  # supprimer les balises HTML
    
    text = re.sub("["
                  u"\U0001F600-\U0001F64F"
                  u"\U0001F300-\U0001F5FF"
                  u"\U0001F680-\U0001F6FF"
                  u"\U0001F1E0-\U0001F1FF"
                  u"\U00002702-\U000027B0"
                  u"\U000024C2-\U0001F251"
                  "]+", ' ', text)  # supprimer les émojis et symboles
    
    text = re.sub(r'@\S+', ' ', text)  # supprimer les mentions @user
    
    text = re.sub(r'#', '', text)  # supprimer le symbole # mais garder le mot
    
    text = re.sub(r'[^a-zA-Z\s]', ' ', text)  # garder uniquement les lettres et les espaces
    
    text = text.lower()  # convertir en minuscules
    
    text = re.sub(r'\s+', ' ', text).strip()  # supprimer les espaces multiples
    
    return text

In [6]:
df["text"] = df["text"].apply(cleaning_simple)

In [7]:
df.head()

,id,keyword,location,text,target
0,0,ablaze,NaN,communal violence in bhainsa telangana stones ...,1
1,1,ablaze,NaN,telangana section has been imposed in bhainsa ...,1
2,2,ablaze,New York City,arsonist sets cars ablaze at dealership,1
3,3,ablaze,"Morgantown, WV",arsonist sets cars ablaze at dealership,1
4,4,ablaze,NaN,lord jesus your love brings freedom and pardon...,0


<h3 style="color: #1e40af; border-bottom: 2px solid #3b82f6; padding-bottom: 8px; margin-top: 25px; font-weight: bold; font-family: 'Segoe UI', sans-serif;">
    Séparation des données
</h3>


In [8]:
# Séparation Train/Test (80% / 20%)
X_train, X_test, y_train, y_test = train_test_split(
    df['text'], df['target'], test_size=0.2, random_state=42, stratify=df['target']
)

# Affiche dans la console le nombre de tweets utilisés pour l'entraînement
print(f"Taille de l'entraînement : {len(X_train)}")
# Affiche dans la console le nombre de tweets gardés pour le test
print(f"Taille du test : {len(X_test)}")

Taille de l'entraînement : 9096
Taille du test : 2274


<h3 style="color: #1e40af; border-bottom: 2px solid #3b82f6; padding-bottom: 8px; margin-top: 25px; font-weight: bold; font-family: 'Segoe UI', sans-serif;">
    Modèles Transformers via Hugging Face
</h3>


In [9]:
# Import de l'objet Dataset de Hugging Face
from datasets import Dataset
# Import de evaluate pour calculer de façon standardisée les métriques d'évaluation
# Import de NumPy
import numpy as np

# Transformation de notre DataFrame d'entraînement Pandas en un objet "Dataset" ultra-optimisé de Hugging Face
hf_train = Dataset.from_pandas(pd.DataFrame({'text': X_train, 'label': y_train}))
# Transformation de notre DataFrame de test Pandas
hf_test = Dataset.from_pandas(pd.DataFrame({'text': X_test, 'label': y_test}))

# Importation de scikit-learn pour calculer facilement le F2-Score et les métriques par classe

# Fonction exécutée à la fin de chaque Epoch par le Trainer pour calculer le score
def compute_metrics(eval_pred):
    # Séparation des probabilités prédites (logits) et des vraies réponses (labels)
    logits, labels = eval_pred
    # L'argmax récupère la classe ayant reçu la plus forte probabilité (0 ou 1)
    predictions = np.argmax(logits, axis=-1)
    
    # Précision et Rappel par classe
    precision_cls = precision_score(labels, predictions, average=None)
    recall_cls = recall_score(labels, predictions, average=None)
    
    # Calcul des métriques globales
    f1 = f1_score(labels, predictions, average="macro")
    f2 = fbeta_score(labels, predictions, beta=2, average="macro")
    accuracy = accuracy_score(labels, predictions)
    
    # Retourner toutes les métriques pour le suivi MLflow (le Trainer ajoutera automatiquement le préfixe "eval_")
    return {
        "f1_macro": f1,
        "f2_score": f2,
        "precision_class_0": precision_cls[0],
        "precision_class_1": precision_cls[1],
        "recall_class_0": recall_cls[0],
        "recall_class_1": recall_cls[1],
        "accuracy": accuracy
    }

# Importation du système d'exploitation
import os
# Paramétrage de la variable d'environnement qui indique à Hugging Face dans quel dossier MLflow il doit écrire
os.environ["MLFLOW_EXPERIMENT_NAME"] = "Disaster_Tweets_Niveau_3_et_4"
# Augmente le délai à 10 minutes pour éviter les erreurs d'upload sur DagsHub
os.environ["MLFLOW_HTTP_REQUEST_TIMEOUT"] = "600"


In [10]:
# Import des AutoClasses (la magie de Hugging Face pour importer n'importe quel modèle du web en 1 ligne)
from transformers import AutoTokenizer, AutoModelForSequenceClassification, TrainingArguments, Trainer

# Définition d'une fonction Python réutilisable pour entraîner n'importe quel Transformer sans réécrire le code
def train_hf_model(model_id, run_name, batch_size=16, epochs=2):
    print(f"========== Début de l'entraînement pour {model_id} ==========")
    
    # 1. Chargement du Tokenizer spécifique au modèle (le dictionnaire qui transforme les mots en IDs)
    tokenizer = AutoTokenizer.from_pretrained(model_id)
    
    # Fonction qui applique le tokenizer sur une phrase
    def tokenize_function(examples):
        # On coupe les phrases (truncation=True) à 128 "tokens" maximum et on ajoute du vide (padding) pour les plus courtes
        return tokenizer(examples['text'], padding="max_length", truncation=True, max_length=128)
    
    # Application massive et extrêmement rapide (batched=True) du Tokenizer sur tout le jeu d'entraînement
    tokenized_train = hf_train.map(tokenize_function, batched=True)
    # Même chose pour le test
    tokenized_test = hf_test.map(tokenize_function, batched=True)
    
    # 2. Chargement de l'architecture du Transformer avec une tête de classification pour 2 sorties (0 ou 1)
    model = AutoModelForSequenceClassification.from_pretrained(model_id, num_labels=2)
    
    # 3. Paramètres de l'entraînement 
    training_args = TrainingArguments(
        output_dir=f"./results_{run_name}",  # Dossier de sauvegarde
        eval_strategy="epoch",         # Evaluer le modèle à la fin de chaque Epoch
        save_strategy="epoch",               # Sauvegarder un "point de contrôle" à la fin de chaque Epoch
        learning_rate=2e-5,                  # Taux d'apprentissage très petit (spécifique aux Transformers)
        per_device_train_batch_size=batch_size, # Taille des paquets envoyés à la carte graphique (entraînement)
        per_device_eval_batch_size=batch_size,  # Taille des paquets envoyés à la carte graphique (test)
        num_train_epochs=epochs,             # Nombre total d'itérations
        weight_decay=0.01,                   # Ajout de pénalités pour éviter le surapprentissage
        load_best_model_at_end=True,         # A la fin, on recharge la version qui a eu le meilleur score
        report_to="mlflow",                  # Dit au système d'envoyer tout le suivi de cet entraînement vers MLflow (DagsHub)
        run_name=run_name,                   # Nom du run dans l'interface MLflow
    )
    
    # 4. L'Objet Trainer qui s'occupe de gérer toute la boucle mathématique PyTorch en arrière-plan
    trainer = Trainer(
        model=model,
        args=training_args,
        train_dataset=tokenized_train,
        eval_dataset=tokenized_test,
        compute_metrics=compute_metrics, # On utilise notre fonction personnalisée pour mesurer le F1-Score
    )
    
    # Démarre l'entraînement intensif (ou reprend s'il y a un checkpoint)
    import os
    checkpoint_dir = f"./results_{run_name}"
    resume = False
    if os.path.exists(checkpoint_dir) and os.listdir(checkpoint_dir):
        resume = True
        print(f"--- Reprise de l'entraînement à partir du dernier checkpoint dans {checkpoint_dir} ---")
    
    trainer.train(resume_from_checkpoint=resume)

    # --- NOUVEAU : Sauvegarde locale de sécurité ---
    save_path = f"./best_model_{run_name}"
    print(f"Sauvegarde du modèle en local dans {save_path}...")
    trainer.save_model(save_path)
    tokenizer.save_pretrained(save_path)

    
    try:
        print("Tentative d'envoi du modèle vers DagsHub...")
        components = {"model": trainer.model, "tokenizer": tokenizer}
        mlflow.transformers.log_model(
            transformers_model=components, 
            artifact_path="model",
            task="text-classification"
        )
        print("✅ Modèle enregistré avec succès sur DagsHub !")
        # Tente d'enregistrer le modèle HuggingFace dans MLflow pour la mise en production
    except Exception as e:
        print(f"⚠️ L'envoi vers DagsHub a échoué (souvent dû à la taille du fichier) : {e}")
        print(f"Pas de souci, ton modèle est bien sauvegardé ici : {save_path}")
        print("Avertissement: L'enregistrement du modèle Transformers dans MLflow a échoué:", e)
    
    # Force MLflow à fermer proprement la session de suivi de ce run
    mlflow.end_run()
    # Affiche la fin dans la console
    print(f"========== Fin de l'entraînement pour {model_id} ==========\n")
    return trainer, tokenized_test



<h3 style="color: #1e40af; border-bottom: 2px solid #3b82f6; padding-bottom: 8px; margin-top: 25px; font-weight: bold; font-family: 'Segoe UI', sans-serif;">
    Modèle BERT (Bidirectional Encoder Representations from Transformers)
</h3>

<h3 style="color: #1e40af; border-bottom: 2px solid #3b82f6; padding-bottom: 8px; margin-top: 25px; font-weight: bold; font-family: 'Segoe UI', sans-serif;">
    # **Description du modèle
</h3>
Le modèle révolutionnaire publié par Google en 2018 qui a changé l'histoire du NLP.

<h3 style="color: #1e40af; border-bottom: 2px solid #3b82f6; padding-bottom: 8px; margin-top: 25px; font-weight: bold; font-family: 'Segoe UI', sans-serif;">
    # **Explication du fonctionnement
</h3>
Contrairement aux anciens modèles qui lisaient le texte de gauche à droite, BERT utilise le **Mécanisme d'Attention**. Il regarde *tous* les mots de la phrase simultanément pour comprendre la relation exacte de chaque mot par rapport à tous les autres mots. (Le fameux "Contexte Bidirectionnel Profond"). Il a été pré-entraîné en lisant tout Wikipédia.


In [ ]:
trainer, tokenized_test = train_hf_model(model_id="vinai/bertweet-base", run_name="BERTweet", epochs=2)



========== Début de l'entraînement pour vinai/bertweet-base ==========


Map:   0%|          | 0/9096 [00:00<?, ? examples/s]

Map:   0%|          | 0/2274 [00:00<?, ? examples/s]

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

[transformers] RobertaForSequenceClassification LOAD REPORT from: vinai/bertweet-base
Key                         | Status     | 
----------------------------+------------+-
lm_head.dense.bias          | UNEXPECTED | 
lm_head.layer_norm.bias     | UNEXPECTED | 
roberta.pooler.dense.bias   | UNEXPECTED | 
lm_head.decoder.weight      | UNEXPECTED | 
lm_head.decoder.bias        | UNEXPECTED | 
lm_head.dense.weight        | UNEXPECTED | 
roberta.pooler.dense.weight | UNEXPECTED | 
lm_head.layer_norm.weight   | UNEXPECTED | 
lm_head.bias                | UNEXPECTED | 
classifier.dense.bias       | MISSING    | 
classifier.out_proj.weight  | MISSING    | 
classifier.out_proj.bias    | MISSING    | 
classifier.dense.weight     | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

2026/05/09 05:09:35 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.


Tentative d'envoi du modèle vers DagsHub...


Writing model shards:   0%|          | 0/2 [00:00<?, ?it/s]

2026/05/09 05:09:40 WARNING mlflow.utils.environment: On Windows, timeout is not supported for model requirement inference. Therefore, the operation is not bound by a timeout and may hang indefinitely. If it hangs, please consider specifying the signature manually.


In [ ]:
from transformers import AutoTokenizer, AutoModelForSequenceClassification, Trainer

# Chemin vers la sauvegarde locale
local_model_path = "./best_model_BERTweet"

# Chargement manuel du tokenizer et du modèle
tokenizer = AutoTokenizer.from_pretrained(local_model_path)
model = AutoModelForSequenceClassification.from_pretrained(local_model_path)

# Re-création de l'objet Trainer (pour pouvoir utiliser trainer.predict sur tokenized_test)
trainer = Trainer( 
    model=model,
    compute_metrics=compute_metrics
)

print("Modèle BERT chargé manuellement avec succès depuis le stockage local.")

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

Modèle BERT chargé manuellement avec succès depuis le stockage local.


<h3 style="color: #1e40af; border-bottom: 2px solid #3b82f6; padding-bottom: 8px; margin-top: 25px; font-weight: bold; font-family: 'Segoe UI', sans-serif;">
    Upload manuel sur Mlflow
</h3>


In [20]:
import mlflow.transformers

mlflow.end_run()
# 1. On s'assure d'utiliser le bon dossier d'expérience
mlflow.set_experiment("Disaster_Tweets")

# 2. On lance un run manuel pour l'enregistrement
with mlflow.start_run(run_name="BERTweet_model_manuel"):
    print("Tentative d'envoi manuel du modèle vers DagsHub (cela peut prendre quelques minutes)...")
    
    # Préparation des composants du modèle
    components = {
        "model": model, 
        "tokenizer": tokenizer
    }
    
    # Enregistrement du modèle
    mlflow.transformers.log_model(
        transformers_model=components,
        artifact_path="model",
        task="text-classification"
    )
    
    print("✅ Modèle enregistré avec succès sur MLflow/DagsHub !")


2026/05/09 05:38:54 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.


Tentative d'envoi manuel du modèle vers DagsHub (cela peut prendre quelques minutes)...


Writing model shards:   0%|          | 0/2 [00:00<?, ?it/s]

2026/05/09 05:38:55 WARNING mlflow.transformers: The model card could not be retrieved from the hub due to Repo id must use alphanumeric chars, '-', '_' or '.'. The name cannot start or end with '-' or '.' and the maximum length is 96: './best_model_BERTweet'.
2026/05/09 05:38:55 WARNING mlflow.transformers: Unable to find license information for this model. Please verify permissible usage for the model you are storing prior to use.
2026/05/09 05:38:55 WARNING mlflow.utils.environment: On Windows, timeout is not supported for model requirement inference. Therefore, the operation is not bound by a timeout and may hang indefinitely. If it hangs, please consider specifying the signature manually.


✅ Modèle enregistré avec succès sur MLflow/DagsHub !
🏃 View run BERTweet_model_manuel at: https://dagshub.com/Oscar-AS/disaster-tweets-project.mlflow/#/experiments/4/runs/f71175808e4043778925ac847e3ff83e
🧪 View experiment at: https://dagshub.com/Oscar-AS/disaster-tweets-project.mlflow/#/experiments/4


In [12]:
import matplotlib.pyplot as plt
from sklearn.metrics import precision_recall_curve, auc

# 1. Obtenir les prédictions et probabilités
preds = trainer.predict(tokenized_test)
logits = preds.predictions
y_probs = tf.nn.softmax(logits, axis=-1).numpy()[:, 1]
y_true = preds.label_ids

# 2. Courbe Précision-Rappel
precision, recall, _ = precision_recall_curve(y_true, y_probs)
pr_auc = auc(recall, precision)

plt.figure(figsize=(8, 6))
plt.plot(recall, precision, label=f'PR Curve (AUC = {pr_auc:.2f})', color='b', lw=2)
plt.xlabel('Recall')
plt.ylabel('Precision')
plt.title('Courbe Précision-Rappel (BERT)')
plt.legend(loc='lower left')
plt.grid(True)
plt.show()

c:\Users\Dell\AppData\Local\Programs\Python\Python312\Lib\site-packages\torch\utils\data\dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


ModuleNotFoundError: No module named 'IPython.core.pylabtools'

In [ ]:
import matplotlib.pyplot as plt

# 3. Courbe de Lift
data = pd.DataFrame({'y_true': y_true, 'y_prob': y_probs})
data = data.sort_values(by='y_prob', ascending=False)
data['cumulative_data_fraction'] = np.arange(1, len(data) + 1) / len(data)
data['cumulative_positive_rate'] = data['y_true'].cumsum() / data['y_true'].sum()
data['lift'] = data['cumulative_positive_rate'] / data['cumulative_data_fraction']

plt.figure(figsize=(8, 6))
plt.plot(data['cumulative_data_fraction'], data['lift'], label='Lift Curve', color='orange', lw=2)
plt.axhline(y=1, color='r', linestyle='--', label='Baseline (Random)')
plt.xlabel('Fraction of data')
plt.ylabel('Lift')
plt.title('Courbe de Lift (BERT)')
plt.legend(loc='upper right')
plt.grid(True)
plt.show()

<h3 style="color: #1e40af; border-bottom: 2px solid #3b82f6; padding-bottom: 8px; margin-top: 25px; font-weight: bold; font-family: 'Segoe UI', sans-serif;">
    Interprétabilité du modèle
</h3>

L'interprétabilité permet de comprendre pourquoi le modèle a pris une décision. Nous allons utiliser deux outils :
1. **SHAP** : Pour quantifier l'importance mathématique de chaque mot (contribution positive ou négative).
2. **Transformers Interpret** : Pour visualiser l'influence des mots sous forme de carte de chaleur (heatmap) très intuitive.

<h3 style="color: #1e40af; border-bottom: 2px solid #3b82f6; padding-bottom: 8px; margin-top: 25px; font-weight: bold; font-family: 'Segoe UI', sans-serif;">
    # **1. SHAP (SHapley Additive exPlanations)
</h3>


In [ ]:
import shap
from transformers import pipeline

# Création d'un pipeline de classification pour SHAP
# On utilise le modèle et le tokenizer déjà chargés
clf_pipeline = pipeline("text-classification", model=model, tokenizer=tokenizer, top_k=None)

# Initialisation de l'explainer SHAP spécifique aux modèles de texte
explainer = shap.Explainer(clf_pipeline)

# On prend un échantillon du test set pour l'explication globale et locale
sample_tweets = X_test.iloc[:10].tolist()
shap_values = explainer(sample_tweets)

# 1. Visualisation locale : Pourquoi ce tweet spécifique est classé comme tel ?
print("Analyse du premier tweet de l'échantillon :")
# On cherche l'index de la classe 'LABEL_1' (Disaster)
# Dans un pipeline text-classification, c'est souvent LABEL_1 ou Disaster
shap.plots.text(shap_values[0, :, "LABEL_1"])

In [ ]:
# 2. Visualisation globale : Quels sont les mots qui pèsent le plus sur l'ensemble de l'échantillon ?

shap.plots.bar(shap_values[:, :, "LABEL_1"].mean(0))

<h3 style="color: #1e40af; border-bottom: 2px solid #3b82f6; padding-bottom: 8px; margin-top: 25px; font-weight: bold; font-family: 'Segoe UI', sans-serif;">
    # **2. Transformers Interpret (Heatmaps visuelles)
</h3>


In [ ]:
from transformers_interpret import SequenceClassificationExplainer

# Initialisation de l'explainer (plus simple et très visuel)
cls_explainer = SequenceClassificationExplainer(model, tokenizer)

# Test sur un tweet spécifique (ex: un vrai tweet de catastrophe ou un piège)
test_tweet = X_test.iloc[0] # Ou n'importe quel texte personnalisé
word_attributions = cls_explainer(test_tweet)

print(f"Tweet analysé : {test_tweet}")
print(f"Classe prédite : {cls_explainer.predicted_class_name}")

# Affichage de la Heatmap
cls_explainer.visualize()